# Model comparison on identical validation rows

Each model is trained in its own original setup (its own folds). Every prediction is out-of-fold:
it comes from that model's fold that never saw the patient. Rows are matched by identity
(eeg_id + exact vote counts), so the two models may use different splits. Reports:
1. KLD — pooled and mean-of-folds, on the ≥10-vote rows and on all rows
2. Paired difference (B − A) with a **patient-level bootstrap** 95% CI (rows from one patient are correlated)
3. Per-class AUROC / recall / precision side by side
4. KLD by vote-count bucket (where does each model win?)
5. A 50/50 ensemble of the two, as a quick upper-bound check

Runs in either kernel (numpy/pandas/sklearn only). Results saved to `results/comparisons/`.

In [ ]:
PROJECT_DIR = '~/hms'
MODEL_A = 'vipeegnet_groupkfold_v1'
MODEL_B = 'effnetb0_native_v1'
N_BOOT = 2000

In [ ]:
import os, sys, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix
P = os.path.abspath(os.path.expanduser(PROJECT_DIR))
sys.path.insert(0, f'{P}/results'); os.environ['HMS_RESULTS'] = f'{P}/results'
from load_results import load_oof, kld, summary
CLASS = ['Seizure', 'LPD', 'GPD', 'LRDA', 'GRDA', 'Other']
pid = pd.read_csv(f'{P}/hms-data/train.csv').groupby('eeg_id')['patient_id'].first()
OUTD = f'{P}/results/comparisons/{MODEL_A}__vs__{MODEL_B}'; os.makedirs(OUTD, exist_ok=True)

def keyed(tag, subset):
    yt, yp, m = load_oof(tag, subset)
    raw = np.rint(yt * m.votes.values[:, None]).astype(int)          # exact vote counts -> row identity
    m['key'] = m.eeg_id.astype(str) + ':' + pd.Series([','.join(map(str, r)) for r in raw]).values
    assert m.key.is_unique, f'{tag}: duplicate row keys'
    return yt, yp, m

data = {}
for subset in ('hq', 'all'):
    ya, pa, ma = keyed(MODEL_A, subset)
    yb, pb, mb = keyed(MODEL_B, subset)
    assert set(ma.key) == set(mb.key), f'{subset}: models were scored on different rows'
    order = pd.Series(np.arange(len(mb)), index=mb.key)[ma.key].values  # put B in A's row order
    yb, pb, mb = yb[order], pb[order], mb.iloc[order].reset_index(drop=True)
    assert np.allclose(ya, yb, atol=1e-5), f'{subset}: labels differ'
    ma['patient_id'] = ma.eeg_id.map(pid).values
    ma['fold_b'] = mb.fold.values
    data[subset] = (ya, pa, pb, ma)
    same = (ma.fold.values == mb.fold.values).all()
    print(f'{subset}: {len(ya):,} identical rows | {ma.patient_id.nunique()} patients | '
          f'{"same" if same else "different"} fold assignment')

## 1–2. KLD and paired difference

In [ ]:
rng = np.random.default_rng(0)
def patient_bootstrap(d, groups, n=N_BOOT):
    g = pd.Series(d).groupby(groups.values); sums, cnts = g.sum().values, g.size().values
    idx = rng.integers(0, len(sums), (n, len(sums)))
    return sums[idx].sum(1) / cnts[idx].sum(1)

rows = []
for subset, (yt, pa, pb, m) in data.items():
    ka, kb = kld(yt, pa), kld(yt, pb)
    d = kb - ka
    bs = patient_bootstrap(d, m.patient_id)
    fa = m.assign(k=ka).groupby('fold').k.mean(); fb = m.assign(k=kb).groupby('fold_b').k.mean()   # each model's own folds
    rows.append({'subset': subset, 'n_rows': len(yt),
                 f'{MODEL_A} pooled': ka.mean(), f'{MODEL_B} pooled': kb.mean(),
                 f'{MODEL_A} mean-of-folds': fa.mean(), f'{MODEL_B} mean-of-folds': fb.mean(),
                 'B − A (pooled)': d.mean(), 'CI95 low': np.percentile(bs, 2.5), 'CI95 high': np.percentile(bs, 97.5),
                 'B worse on % rows': (d > 0).mean() * 100})
    print(f'{subset}: per-fold A {fa.round(4).tolist()} | B {fb.round(4).tolist()}')
res = pd.DataFrame(rows).set_index('subset')
res.round(4).T

## 3. Per-class metrics (≥10-vote rows)

In [ ]:
yt, pa, pb, m = data['hq']
yti = yt.argmax(1); onehot = (yti[:, None] == np.arange(6)).astype(int)
def per_class(p):
    cm = confusion_matrix(yti, p.argmax(1), labels=range(6))
    with np.errstate(invalid='ignore', divide='ignore'):
        rec, prec = np.diag(cm) / cm.sum(1) * 100, np.diag(cm) / cm.sum(0) * 100
    auc = [roc_auc_score(onehot[:, c], p[:, c]) for c in range(6)]
    return auc, rec, prec
aa, ra, pra = per_class(pa); ab, rb, prb = per_class(pb)
cls = pd.DataFrame({'n': onehot.sum(0), 'AUROC A': aa, 'AUROC B': ab, 'Recall A %': ra, 'Recall B %': rb,
                    'Prec A %': pra, 'Prec B %': prb}, index=CLASS)
cls.round(3)

## 4. KLD by number of expert votes (all rows)

In [ ]:
yt, pa, pb, m = data['all']
bins = pd.cut(m.votes, [0, 1, 2, 5, 9, 14, 100], labels=['1', '2', '3-5', '6-9', '10-14', '15+'])
by = pd.DataFrame({'bucket': bins, 'A': kld(yt, pa), 'B': kld(yt, pb)}).groupby('bucket', observed=True)
votes = by.agg(n=('A', 'size'), A=('A', 'mean'), B=('B', 'mean')); votes['B − A'] = votes.B - votes.A
votes.round(4)

## 5. 50/50 ensemble

In [ ]:
ens = []
for subset, (yt, pa, pb, m) in data.items():
    pe = (pa + pb) / 2
    ens.append({'subset': subset, 'A': kld(yt, pa).mean(), 'B': kld(yt, pb).mean(), '0.5A+0.5B': kld(yt, pe).mean()})
ens = pd.DataFrame(ens).set_index('subset'); ens.round(4)

In [ ]:
res.to_csv(f'{OUTD}/kld_summary.csv'); cls.to_csv(f'{OUTD}/per_class.csv')
votes.to_csv(f'{OUTD}/kld_by_votes.csv'); ens.to_csv(f'{OUTD}/ensemble.csv')
json.dump({'A': MODEL_A, 'B': MODEL_B, 'kld': res.reset_index().to_dict('records')},
          open(f'{OUTD}/summary.json', 'w'), indent=2, default=float)
print('Saved to', OUTD)